## Part 2 Gnerate FACT table 

In [ ]:
#!pip install pandasql

In [ ]:
import pandas as pd
import pandasql as ps

In [ ]:
df = pd.read_csv('data/users.csv', dtype = {'id': 'object'}, sep = "|")

#### FACT Wallet
per currency, per wallet, per user, total USD deposit value, deposit times, active deposit (last deposit within 30/90 days)


In [ ]:
wallet_df = pd.read_csv('data/wallet.csv', dtype = {'user_id': 'object'}, sep = "|")
deposit_df = pd.read_csv('data/deposit.csv', dtype = {'wallet_id': 'object'}, sep = "|")
rate_df = pd.read_csv('data/rates.csv', sep = "|")

In [ ]:
rate_df.head(5)

In [ ]:
rates = ps.sqldf("""
    select 
        id, currency, rate, 
        case when row_num =1 then 1 else 0 end as is_latest,
        created_at
    from
    (select *,
        row_number() over (partition by currency order by created_at desc) as row_num
    from rate_df) base
""")

In [ ]:
rates.head(5)

In [ ]:
deposit_df.head(5)

In [ ]:
wallet_fact = ps.sqldf("""
with 
deposit_summary as
    (select 
        wallet_id,
        count(*) as total_deposits,
        sum(case when status = 'Successful' then 1 else 0 end) as total_successful_deposits,
        sum(case when cast((JulianDay('2025-01-12') - JulianDay(substr(a.created_at,1,10))) as Integer) <=30 then 1 else 0 end) as active_deposit_30days,
        sum(case when cast((JulianDay('2025-01-12') - JulianDay(substr(a.created_at,1,10))) as Integer) <=90 then 1 else 0 end) as active_deposit_90days,
        sum(case when status = 'Successful' then value*rate else 0 end) as total_successful_value_usd
    from deposit_df a
    left join wallet_df b 
    on a.wallet_id= b.id
    left join rates c 
    on b.currency = c.currency
    and c.is_latest = 1
    group by wallet_id)

select 
    a.*,
    total_deposits,
    total_successful_deposits,
    active_deposit_30days,
    active_deposit_90days,
    total_successful_value_usd
from wallet_df a
left join deposit_summary b
on a.id = b.wallet_id
""")


In [ ]:
wallet_fact.head(5)

In [ ]:
deposit_df[deposit_df['wallet_id'] == 'wa9146669647-mvr']

In [ ]:
wallet_fact.to_csv('l2_fact_wallet.csv', sep="|", index = False)

#### FACT User
- user profile
- kyc number, kyc status, kyc level
- wallet number, total successful deposit, total successful deposit USD value
- most_bet_game_type, total game, total bet value, total bet number, total bet profit value



In [ ]:
bets_df = pd.read_csv('data/bets.csv', dtype = {'user_id': 'object'}, sep = "|")
game_df = pd.read_csv('data/game.csv', dtype = {'wallet_id': 'object'}, sep = "|")
kyc_df = pd.read_csv('data/user_kyc.csv', sep = "|")

In [ ]:
kyc_df.head(5)

In [ ]:
bets_df.head(50)

In [ ]:
user_fact = ps.sqldf("""
with kyc_sum as (
    select 
        user_id,
        max(created_at) as last_kyc_date,
        count(*) as total_kycs
    from kyc_df
    group by user_id
),
kyc_ranked as (
    select 
        user_id,
        kyc_level,
        kyc_outcome,
        created_at,
        case when row_num = 1 then 1 else 0 end as is_latest
    from
    (select *,
        row_number() over (partition by user_id order by created_at desc) as row_num
    from kyc_df) base
),


bets_user_per_gt as(
    select
        user_id,
        type as game_type,
        count(distinct game_id) as total_games_this_type,
        sum(case when a.status <> 'cancelled' then 1 else 0 end) as total_bets,
        sum(case when (a.status <> 'cancelled') and 
                        (cast((JulianDay('2025-01-12') - JulianDay(substr(a.created_at,1,10))) as Integer) <=30) then 1 else 0 end) as bets_in_30days,
        sum(case when (a.status <> 'cancelled') and 
                        (cast((JulianDay('2025-01-12') - JulianDay(substr(a.created_at,1,10))) as Integer) <=90) then 1 else 0 end) as bets_in_90days,
        sum(case when a.status <> 'cancelled' then value else 0 end) as total_bet_value,
        sum(case when a.status <> 'cancelled' then profit else 0 end) as total_bet_profit
    from bets_df a
    left join game_df b
    on a.game_id = b.id
    group by user_id, type
),
fav_game as (
    select 
        user_id,
        game_type as fav_game,
        total_bets as fav_game_bets,
        total_bet_value as fav_game_bet_value,
        total_bet_profit as fav_game_profit
    from (
    select *,
        row_number() over (partition by user_id order by total_games_this_type desc) as row_num
    from bets_user_per_gt
    ) ranked
    where row_num = 1
),
bets_user as (
    select 
        user_id,
        sum(total_games_this_type) as total_games,
        sum(total_bets) as total_bets,
        sum(bets_in_30days) as bets_in_30days,
        sum(bets_in_90days) as bets_in_90days,
        sum(total_bet_value) as total_bet_value,
        sum(total_bet_profit) as total_profit
    from bets_user_per_gt
    group by user_id
),
wallet_sum as (
    select user_id,
        count(*) as wallet_num
    from wallet_df
    group by user_id
)

select 
    a.*,
    b.total_kycs,
    b.last_kyc_date,
    c.kyc_outcome,

    d.wallet_num,

    e.fav_game,
    e.fav_game_bets,
    e.fav_game_bet_value,
    e.fav_game_profit,

    total_games,
    total_bets,
    bets_in_30days,
    bets_in_90days,
    total_bet_value,
    total_profit
    
from df a
left join kyc_sum b
on a.id = b.user_id
left join kyc_ranked c
on a.id = c.user_id 
and c.is_latest = 1

left join wallet_sum d
on a.id = d.user_id

left join fav_game e
on a.id = e.user_id
left join bets_user f
on a.id = f.user_id
""")

In [ ]:
user_fact.head(5)

In [ ]:
user_fact.to_csv('l2_fact_user.csv', sep="|", index = False)